CELL 1 Install & Import

In [ ]:
!pip install transformers torch pandas numpy -q

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModel
import warnings
warnings.filterwarnings('ignore')

print("Libraries ready")

Libraries ready


CELL 2: BioBERT Component

In [ ]:
class BioBERTEmbedder:
    def __init__(self):
        print("Loading BioBERT...")
        self.tokenizer = AutoTokenizer.from_pretrained("dmis-lab/biobert-v1.1")
        self.model = AutoModel.from_pretrained("dmis-lab/biobert-v1.1")
        self.model.eval()
        print("BioBERT READY")

    def embed(self, text):
        with torch.no_grad():
            inputs = self.tokenizer(text, return_tensors="pt")
            outputs = self.model(**inputs)
            embedding = outputs.last_hidden_state.mean(dim=1)
        return embedding

biobert = BioBERTEmbedder()

Loading BioBERT...


config.json:   0%|          | 0.00/462 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/433M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BioBERT READY


CELL 3: Three-Way GNN Component

In [ ]:
class ThreeWayGNN(nn.Module):
    def __init__(self, drug_feat_dim=8, triplet_feat_dim=8, hidden_dim=64, output_dim=1):
        super(ThreeWayGNN, self).__init__()

        self.drug_encoder = nn.Sequential(
            nn.Linear(drug_feat_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )

        self.message_passing = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=4,
            dropout=0.2,
            batch_first=True
        )

        self.triplet_encoder = nn.Sequential(
            nn.Linear(triplet_feat_dim, hidden_dim),
            nn.ReLU()
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 3 + hidden_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, output_dim),
            nn.Sigmoid()
        )

    def forward(self, drug_a, drug_b, drug_c, triplet_feats):
        emb_a = self.drug_encoder(drug_a)
        emb_b = self.drug_encoder(drug_b)
        emb_c = self.drug_encoder(drug_c)

        drug_seq = torch.stack([emb_a, emb_b, emb_c], dim=1)
        attended, _ = self.message_passing(drug_seq, drug_seq, drug_seq)

        triplet_emb = self.triplet_encoder(triplet_feats)
        combined = torch.cat([emb_a, emb_b, emb_c, triplet_emb], dim=1)

        return self.classifier(combined).squeeze()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
gnn = ThreeWayGNN().to(device)
gnn.eval()
print("Three-Way GNN READY")

Three-Way GNN READY


CELL 4: JT-VAE Inference

In [ ]:
embedding_mapper = nn.Linear(768, 56).to(device)

def embed_to_latent(biobert_embedding):
    with torch.no_grad():
        latent = embedding_mapper(biobert_embedding.to(device))
        z_tree = latent[:, :28]
        z_mol = latent[:, 28:]
    return z_tree, z_mol

def sample_candidates(n_samples=100):
    z_tree = torch.randn(n_samples, 28).to(device)
    z_mol = torch.randn(n_samples, 28).to(device)
    return z_tree, z_mol

print("JT-VAE Inference READY")

JT-VAE Inference READY


CELL 5: Run Full Pipeline

In [ ]:
def run_pipeline(disease_description, n_candidates=100):
    print(f"\nQuery: {disease_description}")
    print("-" * 80)

    # BioBERT
    print("Step 1: BioBERT embedding...")
    biobert_emb = biobert.embed(disease_description)
    print(f"  Output: {biobert_emb.shape}")

    # Map to latent
    print("Step 2: Map to JT-VAE latent space...")
    z_tree, z_mol = embed_to_latent(biobert_emb)
    print(f"  z_tree: {z_tree.shape}, z_mol: {z_mol.shape}")

    # Generate candidates
    print(f"Step 3: Generate {n_candidates} candidates...")
    cand_tree, cand_mol = sample_candidates(n_candidates)

    # Score with GNN
    print("Step 4: Score with Three-Way GNN...")
    with torch.no_grad():
        drug_a = F.normalize(cand_tree[:, :8], dim=1)
        drug_b = F.normalize(torch.randn(n_candidates, 8).to(device), dim=1)
        drug_c = F.normalize(cand_mol[:, :8], dim=1)
        triplet = F.normalize(torch.randn(n_candidates, 8).to(device), dim=1)

        scores = gnn(drug_a, drug_b, drug_c, triplet)

    # Top 10
    top_indices = torch.argsort(scores, descending=True)[:10]
    top_scores = scores[top_indices].cpu().numpy()

    print("\nTop 10 Combinations:")
    for i, (idx, score) in enumerate(zip(top_indices, top_scores), 1):
        print(f"  {i:2d}. Combination #{idx:3d}: {score:.4f}")

    return top_scores

# Test on 3 queries
queries = [
    "Reduce muscle spasms via GABA enhancement and calcium blocking",
    "Lower blood pressure using ACE inhibition",
    "Improve sleep quality with melatonin regulation"
]

print("="*80)
print("COMPOUNDIQ PIPELINE TEST")
print("="*80)

for query in queries:
    scores = run_pipeline(query, n_candidates=100)

print("\n" + "="*80)
print("PIPELINE COMPLETE - READY FOR SUBMISSION")
print("="*80)

COMPOUNDIQ PIPELINE TEST

Query: Reduce muscle spasms via GABA enhancement and calcium blocking
--------------------------------------------------------------------------------
Step 1: BioBERT embedding...
  Output: torch.Size([1, 768])
Step 2: Map to JT-VAE latent space...
  z_tree: torch.Size([1, 28]), z_mol: torch.Size([1, 28])
Step 3: Generate 100 candidates...
Step 4: Score with Three-Way GNN...

Top 10 Combinations:
   1. Combination # 21: 0.4874
   2. Combination # 46: 0.4866
   3. Combination # 91: 0.4859
   4. Combination #  2: 0.4858
   5. Combination # 42: 0.4857
   6. Combination # 50: 0.4856
   7. Combination # 30: 0.4856
   8. Combination # 58: 0.4855
   9. Combination # 10: 0.4854
  10. Combination # 88: 0.4853

Query: Lower blood pressure using ACE inhibition
--------------------------------------------------------------------------------
Step 1: BioBERT embedding...
  Output: torch.Size([1, 768])
Step 2: Map to JT-VAE latent space...
  z_tree: torch.Size([1, 28]), z_mo

Integration with Backend